# Notebook 3 – Price Prediction Modelling
## Airbnb Capstone | Madrid vs Tokyo

**Input:** `airbnb_clean_final.csv` (output of Notebook 1)

### Structure
1. Setup & imports
2. Feature selection & justification
3. Preprocessing (log-transform, encoding, train/test split)
4. Model 1 – Linear Regression (baseline)
5. Model 2 – Random Forest
6. Model 3 – XGBoost
7. Model comparison & evaluation
8. Feature importance analysis
9. Residual analysis
10. Key findings summary

---
## 1. Setup & Imports

In [1]:
!pip install xgboost

In [2]:
# ── Mount Drive & imports ─────────────────────────────────────────────────────
#from google.colab import drive
#drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Consistent colour palette (matching Notebook 2)
COLORS  = {"Madrid": "#E63946", "Tokyo": "#457B9D"}
PALETTE = ["#E63946", "#457B9D"]

DATA_PATH = "/content/drive/Shareddrives/MBD_Capstone Project_KPMG/1. Data/Outputs/"

print("All libraries loaded successfully!")

All libraries loaded successfully!


---
## 2. Feature Selection & Justification

Based on the EDA findings from Notebook 2, we select features that showed meaningful relationships with price:

| Feature | Justification |
|---|---|
| `city` | Madrid median €105 vs Tokyo €96 — city-level pricing differs significantly |
| `room_type` | Entire home/apt commands highest prices across both cities |
| `accommodates` | Strong positive linear trend with price (Image 17 in EDA) |
| `bedrooms` | Correlated with accommodates and price |
| `beds` | Additional size signal |
| `bathrooms` | Property quality/size proxy |
| `neighbourhood_cleansed` | Location drives pricing (Embajadores, Shinjuku Ku are top areas) |
| `host_is_superhost` | Superhosts charge ~€3 more in Madrid, ~€20 more in Tokyo |
| `instant_bookable` | Instant bookable listings in Madrid charge €37 more |
| `amenities_count` | More amenities → higher perceived value |
| `review_scores_rating` | Quality signal that guests use to justify higher prices |
| `review_scores_location` | Location score directly tied to desirability |
| `number_of_reviews` | Proxy for listing popularity/demand |
| `availability_365` | Supply signal — low availability may indicate high demand |
| `minimum_nights` | Business model signal (short-stay vs long-stay pricing) |

In [3]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH + "airbnb_clean_final.csv")
print(f"Dataset shape: {df.shape}")
print(df["city"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/Shareddrives/MBD_Capstone Project_KPMG/1. Data/Outputs/airbnb_clean_final.csv'

In [ ]:
# ── Select features ───────────────────────────────────────────────────────────
FEATURES = [
    # Location
    "city",
    "neighbourhood_cleansed",
    # Property
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "amenities_count",
    # Host
    "host_is_superhost",
    "instant_bookable",
    # Reviews & ratings
    "review_scores_rating",
    "review_scores_location",
    "number_of_reviews",
    # Availability
    "availability_365",
    "minimum_nights",
]

TARGET = "price"

# Keep only selected features + target
df_model = df[FEATURES + [TARGET]].dropna()
print(f"Modelling dataset shape (after dropping NaNs): {df_model.shape}")
print(f"\nMissing values:\n{df_model.isnull().sum()}")

---
## 3. Preprocessing

In [ ]:
# ── 3.1 Log-transform the target ──────────────────────────────────────────────
# Price is right-skewed (confirmed in EDA). Log-transforming makes it more
# normally distributed, which helps Linear Regression and improves all models.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_model["price"], bins=50, color="#457B9D", edgecolor="white")
axes[0].set_title("Price Distribution (Original)", fontweight="bold")
axes[0].set_xlabel("Price per Night (EUR)")

df_model["log_price"] = np.log1p(df_model["price"])
axes[1].hist(df_model["log_price"], bins=50, color="#E63946", edgecolor="white")
axes[1].set_title("Log(Price) Distribution (Transformed)", fontweight="bold")
axes[1].set_xlabel("log(Price + 1)")

plt.tight_layout()
plt.show()

print(f"Original price skewness:     {df_model['price'].skew():.3f}")
print(f"Log-transformed skewness:    {df_model['log_price'].skew():.3f}")

In [ ]:
# ── 3.2 Encode categorical variables ─────────────────────────────────────────
# We use Label Encoding for tree-based models (RF, XGBoost handle this well)
# and One-Hot Encoding for Linear Regression

CATEGORICALS = ["city", "room_type", "neighbourhood_cleansed"]

# Label encode for tree-based models
df_encoded = df_model.copy()
label_encoders = {}
for col in CATEGORICALS:
    le = LabelEncoder()
    df_encoded[col + "_encoded"] = le.fit_transform(df_model[col])
    label_encoders[col] = le

# One-hot encode for Linear Regression
df_ohe = pd.get_dummies(df_model.drop(columns=["price", "log_price"]),
                        columns=CATEGORICALS, drop_first=True)

print("Label encoding done for tree-based models.")
print(f"One-hot encoded feature matrix shape: {df_ohe.shape}")

In [ ]:
# ── 3.3 Train / Test Split (80/20) ───────────────────────────────────────────
NUMERIC_FEATURES = [
    "accommodates", "bedrooms", "beds", "bathrooms", "amenities_count",
    "host_is_superhost", "instant_bookable",
    "review_scores_rating", "review_scores_location", "number_of_reviews",
    "availability_365", "minimum_nights"
]

ENCODED_CATS = [c + "_encoded" for c in CATEGORICALS]

# Feature matrix for tree-based models
X_tree = df_encoded[NUMERIC_FEATURES + ENCODED_CATS]
y      = df_encoded["log_price"]   # log-transformed target

# Feature matrix for Linear Regression
X_lr   = df_ohe

# Splits — same random_state for reproducibility
X_tree_train, X_tree_test, y_train, y_test = train_test_split(
    X_tree, y, test_size=0.2, random_state=42
)
X_lr_train, X_lr_test, _, _ = train_test_split(
    X_lr, y, test_size=0.2, random_state=42
)

print(f"Training set:   {X_tree_train.shape[0]:,} rows")
print(f"Test set:       {X_tree_test.shape[0]:,} rows")
print(f"Target:         log(price) — will back-transform for evaluation")

---
## 4. Model 1 – Linear Regression (Baseline)

**Why we start here:** Linear Regression is the simplest interpretable model. It assumes a linear relationship between features and log(price). Given the non-linear patterns we observed in the EDA (e.g., price spread at each accommodates level), we expect this to underperform — but it gives us a baseline to beat.

In [ ]:
# ── Train Linear Regression ───────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_lr_train, y_train)

# Predict & back-transform from log scale
y_pred_lr_log = lr.predict(X_lr_test)
y_pred_lr     = np.expm1(y_pred_lr_log)   # inverse of log1p
y_test_orig   = np.expm1(y_test)          # back-transform true values

# Metrics
rmse_lr = np.sqrt(mean_squared_error(y_test_orig, y_pred_lr))
mae_lr  = mean_absolute_error(y_test_orig, y_pred_lr)
r2_lr   = r2_score(y_test_orig, y_pred_lr)

print("── Linear Regression Results ────────────────────────")
print(f"  RMSE : €{rmse_lr:.2f}")
print(f"  MAE  : €{mae_lr:.2f}")
print(f"  R²   : {r2_lr:.4f}")

---
## 5. Model 2 – Random Forest

**Why Random Forest:** Our EDA showed non-linear relationships (e.g., price vs accommodates has a wide spread, room type creates distinct price clusters). Random Forest handles this by building many decision trees and averaging their predictions. It is also robust to outliers, which matters given the right-skewed price distribution we observed.

In [ ]:
# ── Train Random Forest ───────────────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_tree_train, y_train)

# Predict & back-transform
y_pred_rf_log = rf.predict(X_tree_test)
y_pred_rf     = np.expm1(y_pred_rf_log)

# Metrics
rmse_rf = np.sqrt(mean_squared_error(y_test_orig, y_pred_rf))
mae_rf  = mean_absolute_error(y_test_orig, y_pred_rf)
r2_rf   = r2_score(y_test_orig, y_pred_rf)

print("── Random Forest Results ────────────────────────────")
print(f"  RMSE : €{rmse_rf:.2f}")
print(f"  MAE  : €{mae_rf:.2f}")
print(f"  R²   : {r2_rf:.4f}")

---
## 6. Model 3 – XGBoost

**Why XGBoost:** XGBoost (Extreme Gradient Boosting) is consistently one of the best-performing models on tabular data. Unlike Random Forest which builds trees in parallel, XGBoost builds them sequentially — each tree corrects the errors of the previous one. This makes it particularly effective for our dataset where pricing errors may follow systematic patterns (e.g., systematically underpricing large Tokyo apartments).

In [ ]:
# ── Train XGBoost ─────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_tree_train, y_train)

# Predict & back-transform
y_pred_xgb_log = xgb_model.predict(X_tree_test)
y_pred_xgb     = np.expm1(y_pred_xgb_log)

# Metrics
rmse_xgb = np.sqrt(mean_squared_error(y_test_orig, y_pred_xgb))
mae_xgb  = mean_absolute_error(y_test_orig, y_pred_xgb)
r2_xgb   = r2_score(y_test_orig, y_pred_xgb)

print("── XGBoost Results ──────────────────────────────────")
print(f"  RMSE : €{rmse_xgb:.2f}")
print(f"  MAE  : €{mae_xgb:.2f}")
print(f"  R²   : {r2_xgb:.4f}")

---
## 7. Model Comparison & Evaluation

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "RMSE (€)": [rmse_lr, rmse_rf, rmse_xgb],
    "MAE (€)": [mae_lr, mae_rf, mae_xgb],
    "R²": [r2_lr, r2_rf, r2_xgb]
})

print("═" * 55)
print(" MODEL COMPARISON SUMMARY")
print("═" * 55)
print(results.to_string(index=False))
print("═" * 55)
print(f"\n Best Model: {results.loc[results['R²'].idxmax(), 'Model']}")
print(f" Best R²:    {results['R²'].max():.4f}")
print(f" Best RMSE:  €{results['RMSE (€)'].min():.2f}")

In [ ]:
# ── Comparison bar chart ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
models    = ["Linear\nRegression", "Random\nForest", "XGBoost"]
bar_color = ["#A8DADC", "#457B9D", "#E63946"]

# RMSE
axes[0].bar(models, [rmse_lr, rmse_rf, rmse_xgb], color=bar_color, edgecolor="white")
axes[0].set_title("RMSE (€) — Lower is Better", fontweight="bold")
axes[0].set_ylabel("RMSE (€)")
for i, v in enumerate([rmse_lr, rmse_rf, rmse_xgb]):
    axes[0].text(i, v + 0.5, f"€{v:.1f}", ha="center", fontweight="bold")

# MAE
axes[1].bar(models, [mae_lr, mae_rf, mae_xgb], color=bar_color, edgecolor="white")
axes[1].set_title("MAE (€) — Lower is Better", fontweight="bold")
axes[1].set_ylabel("MAE (€)")
for i, v in enumerate([mae_lr, mae_rf, mae_xgb]):
    axes[1].text(i, v + 0.5, f"€{v:.1f}", ha="center", fontweight="bold")

# R²
axes[2].bar(models, [r2_lr, r2_rf, r2_xgb], color=bar_color, edgecolor="white")
axes[2].set_title("R² Score — Higher is Better", fontweight="bold")
axes[2].set_ylabel("R²")
axes[2].set_ylim(0, 1)
for i, v in enumerate([r2_lr, r2_rf, r2_xgb]):
    axes[2].text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")

plt.suptitle("Model Comparison: Linear Regression vs Random Forest vs XGBoost",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicted vs Actual plots ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

preds  = [y_pred_lr, y_pred_rf, y_pred_xgb]
titles = ["Linear Regression", "Random Forest", "XGBoost"]
colors = ["#A8DADC", "#457B9D", "#E63946"]

for ax, pred, title, color in zip(axes, preds, titles, colors):
    ax.scatter(y_test_orig, pred, alpha=0.15, s=5, color=color)
    # Perfect prediction line
    max_val = max(y_test_orig.max(), pred.max())
    ax.plot([0, max_val], [0, max_val], "k--", linewidth=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual Price (€)")
    ax.set_ylabel("Predicted Price (€)")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("Predicted vs Actual Price", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 8. Feature Importance Analysis

Feature importance tells us which variables drive price predictions the most — this is key for the business insights we present to KPMG.

In [ ]:
# ── XGBoost Feature Importance ────────────────────────────────────────────────
feature_names = NUMERIC_FEATURES + ENCODED_CATS
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(importance_df["Feature"], importance_df["Importance"],
               color="#E63946", edgecolor="white")
ax.set_xlabel("Feature Importance (XGBoost)", fontsize=12)
ax.set_title("XGBoost – Feature Importance", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(importance_df.tail(5)[["Feature", "Importance"]].to_string(index=False))

In [ ]:
# ── Random Forest Feature Importance (comparison) ─────────────────────────────
rf_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(rf_importance_df["Feature"], rf_importance_df["Importance"],
             color="#457B9D", edgecolor="white")
axes[0].set_title("Random Forest – Feature Importance", fontweight="bold")
axes[0].set_xlabel("Importance")

axes[1].barh(importance_df["Feature"], importance_df["Importance"],
             color="#E63946", edgecolor="white")
axes[1].set_title("XGBoost – Feature Importance", fontweight="bold")
axes[1].set_xlabel("Importance")

plt.suptitle("Feature Importance Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Residual Analysis

Residuals = Actual − Predicted. A good model has residuals that are randomly distributed around zero with no systematic pattern.

In [ ]:
# ── Residual plots for best model (XGBoost) ───────────────────────────────────
residuals_xgb = y_test_orig - y_pred_xgb

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_xgb, residuals_xgb, alpha=0.2, s=5, color="#E63946")
axes[0].axhline(0, color="black", linewidth=1.5, linestyle="--")
axes[0].set_xlabel("Predicted Price (€)")
axes[0].set_ylabel("Residual (€)")
axes[0].set_title("XGBoost: Residuals vs Predicted", fontweight="bold")

# Residual distribution
axes[1].hist(residuals_xgb, bins=60, color="#E63946", edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1.5, linestyle="--")
axes[1].set_xlabel("Residual (€)")
axes[1].set_ylabel("Count")
axes[1].set_title("XGBoost: Residual Distribution", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Mean residual:   €{residuals_xgb.mean():.2f}  (closer to 0 = less bias)")
print(f"Std of residuals: €{residuals_xgb.std():.2f}")

In [ ]:
# ── Residuals by city ─────────────────────────────────────────────────────────
# Check if the model performs differently for Madrid vs Tokyo
test_idx   = y_test.index
city_test  = df_encoded.loc[test_idx, "city"]

res_madrid = residuals_xgb[city_test == "Madrid"]
res_tokyo  = residuals_xgb[city_test == "Tokyo"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(res_madrid, bins=50, alpha=0.6, color="#E63946", label="Madrid", edgecolor="white")
ax.hist(res_tokyo,  bins=50, alpha=0.6, color="#457B9D", label="Tokyo",  edgecolor="white")
ax.axvline(0, color="black", linewidth=1.5, linestyle="--")
ax.set_xlabel("Residual (€)")
ax.set_ylabel("Count")
ax.set_title("XGBoost: Residuals by City", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Madrid — Mean residual: €{res_madrid.mean():.2f} | MAE: €{res_madrid.abs().mean():.2f}")
print(f"Tokyo  — Mean residual: €{res_tokyo.mean():.2f}  | MAE: €{res_tokyo.abs().mean():.2f}")

---
## 10. Key Findings Summary

In [ ]:
# ── Auto-generated summary ────────────────────────────────────────────────────
best_model = results.loc[results["R²"].idxmax(), "Model"]
best_r2    = results["R²"].max()
best_rmse  = results.loc[results["R²"].idxmax(), "RMSE (€)"]
best_mae   = results.loc[results["R²"].idxmax(), "MAE (€)"]

top_feature = importance_df.iloc[-1]["Feature"]

print("═" * 60)
print(" KEY FINDINGS: PRICE PREDICTION MODEL")
print("═" * 60)

print(f"\n MODELS TRAINED")
print(f"  1. Linear Regression (baseline)")
print(f"  2. Random Forest")
print(f"  3. XGBoost")

print(f"\n BEST MODEL: {best_model}")
print(f"  R²   : {best_r2:.4f}  → explains {best_r2*100:.1f}% of price variance")
print(f"  RMSE : €{best_rmse:.2f}  → average prediction error")
print(f"  MAE  : €{best_mae:.2f}  → median absolute error")

print(f"\n TOP PRICE DRIVER: {top_feature}")

print(f"\n MODEL IMPROVEMENT OVER BASELINE")
print(f"  R² gain (LR → XGBoost): +{(best_r2 - r2_lr):.4f}")
print(f"  RMSE reduction:         €{rmse_lr - best_rmse:.2f}")

print("\n RESIDUAL ANALYSIS")
print(f"  Madrid MAE: €{res_madrid.abs().mean():.2f}")
print(f"  Tokyo  MAE: €{res_tokyo.abs().mean():.2f}")
print("═" * 60)